In [1]:
import numpy as np 
import pandas as pd 
import json

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OpenAI_API")

from openai import OpenAI
client = OpenAI(api_key=secret_value_0)

In [3]:
test_data_df = pd.read_csv("/kaggle/input/notebooks/zygong1994/finetuning-experiment-0326-etl/test.csv")

In [4]:
sys_prompt = '''
    You are an expert sales analyst. 
    Given the following sales conversation between a customer and a sales representative, 
    estimate the probability (0-1) that this customer will convert (i.e., purchase the product/sign up for service)
'''

In [5]:
test_data_df.head()

,conversation_id,conversation,full_text,outcome,text
0,saas-12-conv-4816,"[{""speaker"": ""customer"", ""message"": ""Hey, I sa...","Hey, I saw your post about SmartLLM's integrat...",1,<|system|>You are an expert sales call analyst...
1,saas-1-conv-4852,"[{""speaker"": ""customer"", ""message"": ""Hey, so, ...","Hey, so, I've been looking into some pricing t...",0,<|system|>You are an expert sales call analyst...
2,saas-0-conv-1824,"[{""speaker"": ""customer"", ""message"": ""Hey, than...","Hey, thanks for hopping on this call! So, um, ...",1,<|system|>You are an expert sales call analyst...
3,saas-10-conv-3173,"[{""speaker"": ""sales_rep"", ""message"": ""Hey Jess...","Hey Jessica, hope you're doing well! I wanted ...",0,<|system|>You are an expert sales call analyst...
4,saas-18-conv-466,"[{""speaker"": ""customer"", ""message"": ""Hey, so I...","Hey, so I've been, like, struggling with track...",0,<|system|>You are an expert sales call analyst...


# Batch Inference

In [6]:
json_schema = {
    "name": "conversion_estimate",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "probability": {
                "type": "number",
                "description": "Conversion probability 0-1"
            }
        },
        "required": ["probability"],
        "additionalProperties": False
    }
}

text_strings = []
gpt_probs = []

for idx, row in test_data_df.iterrows():
    convo = json.loads(row['conversation'])
    text = "\n".join(f"{turn['speaker']}: {turn['message']}" for turn in convo)
    text_strings.append(text)
    
    response = client.chat.completions.create(
        model="gpt-4.1-2025-04-14",
        messages=[{"role": "user", "content": sys_prompt + text}],
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": json_schema
        }
    )
    
    result = json.loads(response.choices[0].message.content)
    gpt_probs.append(result['probability'])
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(test_data_df)}")

print(f"Done. Processed {len(test_data_df)} conversations.")

Processed 10/200
Processed 20/200
Processed 30/200
Processed 40/200
Processed 50/200
Processed 60/200
Processed 70/200
Processed 80/200
Processed 90/200
Processed 100/200
Processed 110/200
Processed 120/200
Processed 130/200
Processed 140/200
Processed 150/200
Processed 160/200
Processed 170/200
Processed 180/200
Processed 190/200
Processed 200/200
Done. Processed 200 conversations.


In [7]:
results_df = pd.DataFrame({
    'text_string_conversation': text_strings,
    'ground_truth_conversion': test_data_df['outcome'].values,
    'gpt_probability': gpt_probs
})

results_df.head()

,text_string_conversation,ground_truth_conversion,gpt_probability
0,"customer: Hey, I saw your post about SmartLLM'...",1,0.60
1,"customer: Hey, so, I've been looking into some...",0,0.40
2,"customer: Hey, thanks for hopping on this call...",1,0.60
3,"sales_rep: Hey Jessica, hope you're doing well...",0,0.20
4,"customer: Hey, so I've been, like, struggling ...",0,0.25


In [8]:
results_df.to_csv("GPT_results.csv")

# Evaluation

In [9]:
from sklearn.metrics import roc_auc_score

y_true = results_df['ground_truth_conversion'].values
y_prob = results_df['gpt_probability'].values

# AUC-ROC
auc_roc = roc_auc_score(y_true, y_prob)
print(f"AUC-ROC: {auc_roc:.4f}")

# Cross-entropy: H(delta_y, p) = -[y * log(p) + (1-y) * log(1-p)]
# Clip to avoid log(0)
eps = 1e-15
y_prob_clipped = np.clip(y_prob, eps, 1 - eps)
cross_entropy = -np.mean(y_true * np.log(y_prob_clipped) + (1 - y_true) * np.log(1 - y_prob_clipped))
print(f"Binary Cross-Entropy: {cross_entropy:.4f}")

AUC-ROC: 0.8105
Binary Cross-Entropy: 0.5566
